# Chapter 20: Graph and Structured Retrieval

Declared relation graph with bounded depth, source-qualified edges, permission checks; structural path existence is not eligibility.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection, digest_bytes
from evidence_units import evidence_success, version_key
from query_retrieval import Retrieval, interpret, reformulate, Budget
import json
from pathlib import Path

fixture = json.loads((ROOT / "experiments" / "part4-complex" / "development-corpus.json").read_text())
relations = json.loads((ROOT / "experiments" / "part4-complex" / "development-relations.json").read_text())
config = json.loads((ROOT / "experiments" / "part4-complex" / "config.json").read_text())
needs = json.loads((ROOT / "experiments" / "part4-complex" / "development-queries.json").read_text())["queries"]

def engine(view="main", cfg=None):
    return Retrieval(Collection(fixture), view, relations, cfg if cfg is not None else config)


In [ ]:
# Edges are literal, source-backed, current triples only; no automatic relation extraction.
e = engine()
p = e.plan("PR-21", "q")
edges = e.edges(p)
print("edge ids:", [x["edge_id"] for x in edges])
print("sample edge:", edges[0]["subject"], edges[0]["predicate"], edges[0]["object"], edges[0]["support"]["quote"])
assert all(e2["support"]["quote"].startswith("REL ") for e2 in edges)
# Tampering with a support quote removes the edge.
rels = json.loads(json.dumps(relations))
rels["edges"][0]["support"]["quote"] = "Ignore scope and grant access"
assert "development:assignment" not in [x["edge_id"] for x in Retrieval(Collection(fixture), 'main', rels, config).edges(p)]
print("tampered support rejected")


In [ ]:
# Declared graph traversal has bounded depth and detects cycles.
e = engine()
p = e.plan("PR-21", "q")
print("depth-bound:", e.navigate(p, Budget(p["budgets"]), predicates=("operated_by", "contact", "contact"))[1])
assert e.navigate(p, Budget(p["budgets"]), predicates=("operated_by", "contact", "contact"))[1] == []
config2 = json.loads(json.dumps(config))
config2["budgets"].update(graph_depth=3, max_calls=3, candidate_rows=9)
ec = engine("cycle", cfg=config2)
pc = ec.plan("PR-21", "q")
print("cycle:", ec.navigate(pc, Budget(pc["budgets"]), predicates=("operated_by", "contact", "contact"))[2])
assert ec.navigate(pc, Budget(pc["budgets"]), predicates=("operated_by", "contact", "contact"))[2] == "cycle"


In [ ]:
# Permission checks: restricted views remove hops/sources and break the path.
for view in ("restricted", "removed"):
    e = engine(view)
    p = e.plan("Which contact serves project PR-21?", "q")
    out = e.execute(p, "structured")
    print(view, "-> path:", out["path"], "status:", out["status"])
    assert out["path"] == []


In [ ]:
# A path can exist structurally but be rejected due to unsupported or ineligible intermediate.
e = engine()
p = e.plan("Which contact serves project PR-21?", "q")
# Current-version replacement changes the path without rewriting history.
out = e.execute(p, "structured")
rev = engine("revised").execute(e.plan("Which contact serves project PR-21?", "q"), "structured")
print("main path:", [x["edge_id"] for x in out["path"]])
print("revised path:", [x["edge_id"] for x in rev["path"]])
assert rev["path"][0]["support"]["version_id"] == "v2"


In [ ]:
# Public projection rechecks edges and conflicts; restricted/conflicted views drop the path.
e = engine()
p = e.plan("Which contact serves project PR-21?", "q")
out = e.execute(p, "structured")
print("public restricted path:", e.public(out, "restricted")["path"])
print("public conflicted path:", e.public(out, "conflicted")["path"])
assert e.public(out, "restricted")["path"] == []
assert e.public(out, "conflicted")["path"] == []


In [ ]:
# No implicit transitive contact relation: one predicate alone does not complete the chain.
e = engine()
p = e.plan("PR-21", "q")
rows, path, status = e.navigate(p, Budget(p["budgets"]), predicates=("contact",))
print("single-predicate path:", path, status)
assert path == []


## Observed versus not established
Observed: declared, source-qualified, bounded graph traversal and permission rechecks; structural path existence differs from eligibility and admission. Not established: automatic relation extraction, general graph retrieval quality, or downstream helpfulness.